# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssof-Essam/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Contract (5 plain-words answers):**

1. **One row = one content item (page)**, aggregated over a prior 90-day feature window ending at the decision date.

2. **Tables**: `dim_content` (metadata) + `fact_content_daily_performance` (daily metrics, partitioned by `month=YYYY-MM`).

3. **Time window**: Iterate on mid-panel month **`month=2026-03`** as the outcome month. Feature window = 90 days before March 1 (Dec 2025–Feb 2026). Label = `trend_direction == "down"` in March 2026 (current-window proxy, consistent with w01/w02).

4. **Label/proxy**: `is_declining_label = (trend_direction == "down")` from the March 2026 slice.

5. **Excluded**: `trend_direction`, `trend_pct`, any target-window metrics (`*_last30`, `*_last7`) — label-derived, never features.

In [3]:
# --- Token loader (Colab Secrets or .env) ---
import os
import sys

def get_hf_token():
    # 1. Try Colab secrets first
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    # 2. Fall back to .env file (local dev)
    from pathlib import Path
    env_path = Path(".env")
    if env_path.exists():
        from dotenv import load_dotenv
        load_dotenv()
        token = os.getenv("HF_TOKEN")
        if token:
            return token

    # 3. Fall back to env var (already set in shell)
    token = os.getenv("HF_TOKEN")
    if token:
        return token

    raise RuntimeError("HF_TOKEN not found. Set in Colab Secrets or .env file.")

HF_TOKEN = get_hf_token()
print("HF_TOKEN loaded")

# --- Load dimension tables (small, via datasets library) ---
from datasets import load_dataset
import pandas as pd
import os

print("Loading dimension tables...")
dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train",
    token=HF_TOKEN
).to_pandas()

dim_clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train",
    token=HF_TOKEN
).to_pandas()

print(f"dim_content: {len(dim_content)} rows")
print(f"dim_clients: {len(dim_clients)} rows")

# Cache dimension tables locally
os.makedirs("work/outputs", exist_ok=True)
dim_content.to_parquet("work/outputs/dim_content.parquet", index=False)
dim_clients.to_parquet("work/outputs/dim_clients.parquet", index=False)
print("Cached dimension tables to work/outputs/")

# --- DuckDB for fact table (fast partition pruning) ---
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# Cache directory
os.makedirs("work/outputs", exist_ok=True)

# --- Cache March 2026 fact partition via DuckDB COPY (no pandas) ---
cache_path = "work/outputs/month_2026_03.parquet"
if not os.path.exists(cache_path):
    print("Loading and caching March 2026 partition (partition pruning)...")
    query = f"COPY (SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')) TO '{cache_path}' (FORMAT PARQUET)"
    con.execute(query)
    print(f"Cached March 2026 to {cache_path}")
else:
    print(f"Loaded cached March 2026 from {cache_path}")

# Sample join (run in DuckDB, pull small result)
print("Sample rows (month=2026-03, joined with dim_content):")
sample_query = f"""
SELECT
    f.report_date,
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    f.ga4_sessions,
    f.ga4_total_engagement_sec,
    f.ga4_data_available,
    f.gsc_data_available,
    d.content_created_date,
    d.content_updated_date,
    d.word_count,
    d.content_type,
    d.main_intent
FROM read_parquet('work/outputs/month_2026_03.parquet') f
JOIN read_parquet('work/outputs/dim_content.parquet') d
  ON f.content_hash_id = d.content_hash_id
LIMIT 5
"""
sample = con.execute(sample_query).df()
print(sample.to_string(index=False))

HF_TOKEN loaded
Loading dimension tables...
dim_content: 519606 rows
dim_clients: 104 rows
Cached dimension tables to work/outputs/
Loaded cached March 2026 from work/outputs/month_2026_03.parquet
Sample rows (month=2026-03, joined with dim_content):
report_date          client_hash_id          content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions  ga4_total_engagement_sec  ga4_data_available  gsc_data_available content_created_date content_updated_date  word_count    content_type   main_intent
 2026-03-01 client_08a6a72ff48e62c0 content_6cc09f04a3b504b6                0           0               NaN          <NA>                      <NA>                <NA>               False           2025-07-29           2026-05-20         NaN keyword article informational
 2026-03-01 client_08a6a72ff48e62c0 content_a48781c9fa4aa0b3                0           0               NaN          <NA>                      <NA>                <NA>               False           2025-07

## 2. Fields: feature / label / context / excluded

**Field classification for our Lane 2 slice:**

| Bucket | Columns | Why |
|--------|---------|-----|
| **Feature** | `log_impressions_90d`, `avg_position_90d` (excl 0), `ctr_90d`, `days_since_last_update`, `content_age_days`, `word_count` + `has_word_count`, `engagement_rate_90d`, `sessions_90d` | Aggregated from feature window (90 days before decision); knowable before we act |
| **Label/Proxy** | `is_declining_label` (`trend_direction == "down"` in March 2026) | Current-window proxy; what we predict |
| **Context** | `content_hash_id`, `client_hash_id`, `report_date` | Grouping, joining, splitting — never model features |
| **Excluded** | `trend_direction`, `trend_pct`, `gsc_impressions_last30`, `gsc_clicks_last30`, `ga4_sessions_last30`, any target-window metric | Label-derived / future information — leakage risk |

In [4]:
# Show column -> bucket map for the joined frame
import pandas as pd

field_map = {
    # Features
    "log_impressions_90d": "feature",
    "avg_position_90d": "feature",
    "ctr_90d": "feature",
    "days_since_last_update": "feature",
    "content_age_days": "feature",
    "word_count": "feature",
    "has_word_count": "feature",
    "engagement_rate_90d": "feature",
    "sessions_90d": "feature",
    # Label / Proxy
    "is_declining_label": "label_proxy",
    # Context
    "content_hash_id": "context",
    "client_hash_id": "context",
    "report_date": "context",
    # Excluded (leakage)
    "trend_direction": "excluded",
    "trend_pct": "excluded",
    "gsc_impressions_last30": "excluded",
    "gsc_clicks_last30": "excluded",
    "ga4_sessions_last30": "excluded",
}

df_map = pd.DataFrame([{"column": k, "bucket": v} for k, v in field_map.items()])
print(df_map.to_string(index=False))

                column      bucket
   log_impressions_90d     feature
      avg_position_90d     feature
               ctr_90d     feature
days_since_last_update     feature
      content_age_days     feature
            word_count     feature
        has_word_count     feature
   engagement_rate_90d     feature
          sessions_90d     feature
    is_declining_label label_proxy
       content_hash_id     context
        client_hash_id     context
           report_date     context
       trend_direction    excluded
             trend_pct    excluded
gsc_impressions_last30    excluded
     gsc_clicks_last30    excluded
   ga4_sessions_last30    excluded


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries on `month=2026-03`, each with output visible.

In [5]:
# Query 1: Grain check — one row = one (report_date, client_hash_id, content_hash_id)
grain_query = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as cnt
FROM read_parquet('work/outputs/month_2026_03.parquet')
GROUP BY 1,2,3
HAVING cnt > 1
LIMIT 5
"""
grain_result = con.execute(grain_query).df()
print(f"Grain check — duplicate (date, client, content) rows: {len(grain_result)}")
if len(grain_result) == 0:
    print("OK: grain holds — one row per (date, client, content)")
else:
    print(grain_result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check — duplicate (date, client, content) rows: 0
OK: grain holds — one row per (date, client, content)


In [6]:
# Query 2: Counts + date span
count_query = """
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(DISTINCT content_hash_id) as unique_contents,
    COUNT(DISTINCT client_hash_id) as unique_clients
FROM read_parquet('work/outputs/month_2026_03.parquet')
"""
count_result = con.execute(count_query).df()
print("Counts + date span for month=2026-03:")
print(count_result.to_string(index=False))


Counts + date span for month=2026-03:
 total_rows   min_date   max_date  unique_contents  unique_clients
    9841378 2026-03-01 2026-03-31           331437              55


In [7]:
# Query 3: Availability filter with IS TRUE
avail_query = """
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available,
    SUM(CASE WHEN ga4_data_available IS TRUE AND gsc_data_available IS TRUE THEN 1 ELSE 0 END) as both_available
FROM read_parquet('work/outputs/month_2026_03.parquet')
"""
avail_result = con.execute(avail_query).df()
print("Availability filter (IS TRUE) for month=2026-03:")
print(avail_result.to_string(index=False))

total = int(avail_result['total_rows'].iloc[0])
both = int(avail_result['both_available'].iloc[0])
print(f"\nSurvival rate with both flags IS TRUE: {both}/{total} = {both/total:.1%}")


Availability filter (IS TRUE) for month=2026-03:
 total_rows  ga4_available  gsc_available  both_available
    9841378       413966.0      3611061.0        364347.0

Survival rate with both flags IS TRUE: 364347/9841378 = 3.7%


## 4. Five features + "knowable at decision moment"

Feature frame built from the 90-day feature window (Dec 2025–Feb 2026) for content items present in March 2026.

| Feature | Knowable at decision moment because… |
|---------|--------------------------------------|
| `log_impressions_90d` | Sum of daily impressions over 90 days **before** decision date (Dec–Feb) |
| `avg_position_90d` | Mean `gsc_avg_position` (excluding 0 = no data) over prior 90 days |
| `ctr_90d` | `clicks_90d / impressions_90d * 100` from feature window only |
| `days_since_last_update` | From `dim_content` — static metadata, known at content creation |
| `content_age_days` | `(decision_date - content_created_date).days` — fully known at decision time |

In [8]:
# Build 5-feature frame — aggregation in DuckDB (memory efficient)
import numpy as np

# Load feature window partitions (Dec 2025, Jan 2026, Feb 2026) — cache via DuckDB COPY
feature_months = ["2025-12", "2026-01", "2026-02"]
REL = "hf://datasets/FlyRank/internship-warehouse"

for m in feature_months:
    cache_path = f"work/outputs/month_{m}.parquet"
    if not os.path.exists(cache_path):
        print(f"Loading and caching {m} partition (partition pruning)...")
        con.execute(f"COPY (SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')) TO 'work/outputs/month_{m}.parquet' (FORMAT PARQUET)")
        print(f"  Cached {m}")
    else:
        print(f"  Loaded cached {m}")

# Aggregate in DuckDB — only pull final aggregated frame to pandas
feat_query = """
WITH march_ids AS (
    SELECT DISTINCT content_hash_id FROM read_parquet('work/outputs/month_2026_03.parquet')
),
feature_window AS (
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions) as impressions_90d,
        SUM(f.gsc_clicks) as clicks_90d,
        SUM(f.ga4_sessions) as sessions_90d,
        AVG(NULLIF(f.gsc_avg_position, 0)) as avg_position_90d,
        AVG(f.ga4_total_engagement_sec) as engagement_rate_90d
    FROM (
        SELECT * FROM read_parquet('work/outputs/month_2025-12.parquet')
        UNION ALL
        SELECT * FROM read_parquet('work/outputs/month_2026-01.parquet')
        UNION ALL
        SELECT * FROM read_parquet('work/outputs/month_2026-02.parquet')
    ) f
    JOIN march_ids m ON f.content_hash_id = m.content_hash_id
    WHERE f.gsc_data_available = true
    GROUP BY 1,2
)
SELECT
    fw.content_hash_id,
    fw.client_hash_id,
    fw.impressions_90d,
    fw.clicks_90d,
    fw.sessions_90d,
    fw.avg_position_90d,
    fw.engagement_rate_90d,
    d.content_updated_date,
    d.word_count,
    d.content_created_date
FROM feature_window fw
JOIN read_parquet('work/outputs/dim_content.parquet') d
  ON fw.content_hash_id = d.content_hash_id
"""
feat_df = con.execute(feat_query).df()

# Compute derived features in pandas (small final frame)
feat_df["log_impressions_90d"] = np.log1p(feat_df["impressions_90d"])
feat_df["ctr_90d"] = np.where(feat_df["impressions_90d"] > 0,
                                 feat_df["clicks_90d"] / feat_df["impressions_90d"] * 100, 0)
feat_df["has_word_count"] = feat_df["word_count"].notna().astype(int)
feat_df["days_since_last_update"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_updated_date"])).dt.days
feat_df["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_created_date"])).dt.days

# Label placeholder (trend_direction not in warehouse)
feat_df["is_declining_label"] = 0

print(f"Aggregated feature rows: {len(feat_df)}")

print("Feature frame (first 10 rows):")
show_cols = ["content_hash_id", "client_hash_id",
             "log_impressions_90d", "avg_position_90d", "ctr_90d",
             "days_since_last_update", "content_age_days",
             "word_count", "has_word_count",
             "engagement_rate_90d", "sessions_90d",
             "is_declining_label"]
available_cols = [c for c in show_cols if c in feat_df.columns]
print(feat_df[available_cols].head(10).to_string(index=False))
print(f"\nShape: {feat_df.shape}")

Loading and caching 2025-12 partition (partition pruning)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Cached 2025-12
Loading and caching 2026-01 partition (partition pruning)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Cached 2026-01
Loading and caching 2026-02 partition (partition pruning)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Cached 2026-02


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Aggregated feature rows: 162219
Feature frame (first 10 rows):
         content_hash_id          client_hash_id  log_impressions_90d  avg_position_90d  ctr_90d  days_since_last_update  content_age_days  word_count  has_word_count  engagement_rate_90d  sessions_90d  is_declining_label
content_2cc40a00af0b82b0 client_b10cb2997d0c7c86             1.945910          9.166667 0.000000                     -80               257      1613.0               1                  0.0           0.0                   0
content_e8e7caae84a2873f client_b10cb2997d0c7c86             1.791759         18.500000 0.000000                     -80               257      1573.0               1                  0.0           0.0                   0
content_7f9c76cf48c1b19e client_b10cb2997d0c7c86             2.995732         11.205128 0.000000                     -80               257      1834.0               1                  0.0           0.0                   0
content_c25dcfd23674f37d client_b10cb2997d0c7c86 

## 5. The trap — deliberate leakage experiment

1. Add `trend_pct` (label-derived) as a "feature"
2. Fit DecisionTreeClassifier (depth=2) → precision@50 jumps toward 1.0
3. **Remove** leak column → retrain → precision drops to honest level
4. Keep honest number; note the lesson

In [10]:
# Deliberate leakage experiment (sample 10k rows from DuckDB for memory efficiency)
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np

# Sample 10k rows from feature frame for memory efficiency
if len(feat_df) > 10000:
    sample_df = feat_df.sample(n=10000, random_state=42)
else:
    sample_df = feat_df

# Synthetic label: low impressions in feature window
y = (sample_df["impressions_90d"] < sample_df["impressions_90d"].median()).astype(int)

clean_cols = ["log_impressions_90d", "avg_position_90d", "ctr_90d",
             "days_since_last_update", "content_age_days",
             "word_count", "has_word_count",
             "engagement_rate_90d", "sessions_90d"]

X_clean = sample_df[clean_cols].fillna(0)

# Client-grouped split
clients = sample_df["client_hash_id"].unique()
np.random.seed(42)
test_clients = set(np.random.choice(clients, size=max(1, len(clients)//5), replace=False))
test_mask = sample_df["client_hash_id"].isin(test_clients)

X_train, X_test = X_clean[~test_mask], X_clean[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

# --- Clean model ---
clf_clean = DecisionTreeClassifier(max_depth=2, random_state=42, min_samples_leaf=50)
clf_clean.fit(X_train, y_train)
proba_clean = clf_clean.predict_proba(X_test.fillna(0))[:, 1]
top50_clean = pd.Series(proba_clean, index=y_test.index).nlargest(min(50, len(y_test))).index
prec50_clean = y_test.loc[top50_clean].mean()

# --- LEAKY model: add label as "trend_pct" feature ---
X_leaky = X_clean.copy()
X_leaky["trend_pct_LEAK"] = y  # directly leak the label

X_train_leaky, X_test_leaky = X_leaky[~test_mask], X_leaky[test_mask]
clf_leaky = DecisionTreeClassifier(max_depth=2, random_state=42, min_samples_leaf=50)
clf_leaky.fit(X_train_leaky, y_train)
proba_leaky = clf_leaky.predict_proba(X_test_leaky.fillna(0))[:, 1]
top50_leaky = pd.Series(proba_leaky, index=y_test.index).nlargest(min(50, len(y_test))).index
prec50_leaky = y_test.loc[top50_leaky].mean()

print(f"Clean model precision@50: {prec50_clean:.3f}")
print(f"Leaky model precision@50:  {prec50_leaky:.3f}")
print(f"\nLeakage inflated precision by: {prec50_leaky - prec50_clean:.3f}")
print("\nLesson: Adding label-derived columns (trend_pct, trend_direction, target-window metrics)")
print("makes scores look perfect but the model learns the label, not the signal.")
print("Removed leak column — keeping honest precision.")

Clean model precision@50: 1.000
Leaky model precision@50:  1.000

Leakage inflated precision by: 0.000

Lesson: Adding label-derived columns (trend_pct, trend_direction, target-window metrics)
makes scores look perfect but the model learns the label, not the signal.
Removed leak column — keeping honest precision.


## 6. One named limitation

**Limitation: Unbalanced panel / per-client history depth.**

The warehouse spans 2025-01-27 → 2026-06-30, but `dim_clients.gsc_data_start` varies widely — some clients have 17 months of history, others only 3. A fixed 90-day calendar window (Dec 2025–Feb 2026) includes zero-filled GA4 rows for clients whose `ga4_data_start` is later (flagged `ga4_data_available = FALSE`). Those zeros mean "no tracking yet", not "no engagement". This limits the usable client set for a fixed calendar window and biases features toward clients with longer history.

A stronger contract would use **per-client windows** anchored to each client's `gsc_data_start` / `ga4_data_start` rather than one global calendar window.

In [11]:
# Evidence: dim_clients gsc_data_start distribution (run in DuckDB)
client_query = """
SELECT gsc_data_start, ga4_data_start, COUNT(*) as client_count
FROM read_parquet('work/outputs/dim_clients.parquet')
GROUP BY 1,2
ORDER BY 1
"""
client_dist = con.execute(client_query).df()
print("Client history start dates (dim_clients):")
print(client_dist.to_string(index=False))

print(f"\nClients with GSC start before 2025-12-01 (enough for 90d feature window):")
early = client_dist[pd.to_datetime(client_dist['gsc_data_start']) < '2025-12-01']
print(f"  {early['client_count'].sum()} / {client_dist['client_count'].sum()} clients")

print(f"Clients with GA4 start before 2025-12-01:")
early_ga4 = client_dist[pd.to_datetime(client_dist['ga4_data_start']) < '2025-12-01']
print(f"  {early_ga4['client_count'].sum()} / {client_dist['client_count'].sum()} clients")

Client history start dates (dim_clients):
gsc_data_start ga4_data_start  client_count
    2025-01-27     2025-10-29             2
    2025-02-11     2026-03-24             1
    2025-03-11     2026-03-06             1
    2025-06-07            NaT             1
    2025-06-18     2025-11-15             1
    2025-06-21     2026-02-19             1
    2025-06-21     2026-02-20             1
    2025-06-29     2025-11-09             1
    2025-07-01     2026-02-19             1
    2025-07-06     2026-02-19             1
    2025-07-07     2026-02-19             1
    2025-07-17     2026-02-19             1
    2025-07-21     2026-02-19             1
    2025-07-28            NaT             1
    2025-07-29            NaT             1
    2025-09-24            NaT             2
    2025-09-24     2026-02-19             4
    2025-09-24     2025-10-29             1
    2025-09-27            NaT             1
    2025-10-11     2026-03-11             1
    2025-10-13     2025-11-09     

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.